## Limpieza de 'DF_RACERESULT_SUCIO.csv'

RaceResult es distinto a las fuentes anteriores en un aspecto clave: el catálogo de eventos (`DF_RACERESULT_SUCIO.csv`, 759 filas, una por evento) y las clasificaciones (`raceresult_clasificaciones.csv`) se han scrapeado por separado. **De momento las clasificaciones se han descargado para 147 de los 759 eventos del catálogo (19,4%)** — el scraper real (`scrape_raceresult.ipynb`) soporta el resto, pero lanzar la descarga completa de las 612 clasificaciones que aún faltan implica cientos de peticiones contra `my.raceresult.com` y se va ampliando poco a poco corriéndolo en local, no en este entorno de pruebas.

Por eso este notebook deja claro en todo momento **qué filas tienen clasificación real y cuáles no**: `finisher_d`/`finisher_h` solo llevan un número cuando el evento ya tiene clasificación descargada (147 de 759); para el resto van a `NaN` (nunca a 0, que significaría "cero finishers" en vez de "no escrapeado todavía"). Se añade una columna propia `tiene_clasificacion` para que quede explícito al cruzar con Power BI o con las otras fuentes.

Seguimos el mismo proceso que en buscametas/championsxip/carreirasgalegas/ccnorte/cronofinisher/mychip: cargar, diagnosticar, limpiar/renombrar, clasificar `tipo_modalidad` y `publico`, y guardar en el esquema común.

In [1]:
from pathlib import Path
import json as _json
import pandas as pd

DIR_RAW = Path("../../data/raw/raceresult")

eventos = pd.read_csv(DIR_RAW / "DF_RACERESULT_SUCIO.csv")
clasif = pd.read_csv(DIR_RAW / "raceresult_clasificaciones.csv")
with open(DIR_RAW / "raceresult_eventos_sin_resultados.json", encoding="utf-8") as f:
    eventos_sin_resultados = _json.load(f)

print("Catálogo de eventos:", eventos.shape)
print("Filas de clasificación (muestra de validación):", clasif.shape)
print("Eventos con clasificación en la muestra:", sorted(clasif["id_evento"].unique().tolist()))
print("Eventos confirmados SIN resultados publicados:", eventos_sin_resultados)
eventos.head()

Catálogo de eventos: (759, 16)
Filas de clasificación (muestra de validación): (89375, 1513)
Eventos con clasificación en la muestra: [1418, 7123, 13940, 15053, 24234, 97872, 101182, 101221, 101238, 103015, 103018, 103021, 103022, 103023, 103026, 103028, 103389, 103527, 107367, 107985, 109655, 109662, 114698, 118096, 119510, 124038, 124098, 126088, 126408, 127097, 128339, 129506, 129508, 129515, 131126, 141231, 142392, 144010, 147435, 152623, 158824, 160694, 162669, 163782, 169247, 173040, 174566, 177067, 177672, 178840, 180395, 182937, 184010, 263235, 263835, 265092, 265592, 267881, 268450, 268707, 320224, 321332, 321696, 323127, 323691, 324567, 325355, 325768, 328702, 328876, 329092, 329639, 330010, 330439, 332305, 333043, 347670, 348311, 348872, 349109, 349402, 350519, 350594, 350758, 350778, 351661, 352078, 352743, 354574, 354940, 356901, 358201, 359586, 359696, 362250, 377397, 380186, 399628, 402252, 402384, 402389, 402398, 402417, 402421, 402424, 402461, 402563, 402818, 403612, 4

       id  ...       provincia
0  325768  ...      Las Palmas
1  377397  ...          Málaga
2  403612  ...  Islas Baleares
3  263835  ...       La Coruña
4  420299  ...      Pontevedra

[5 rows x 16 columns]

In [2]:
# Diagnóstico antes de limpiar.
print("Valores nulos por columna (catálogo de eventos):")
print(eventos.isna().sum())
print()

print("Filas completamente duplicadas (catálogo):", eventos.duplicated().sum())
print("Filas duplicadas por id:", eventos.duplicated(subset=["id"]).sum())
print()

print("tipo_deporte:")
print(eventos["tipo_deporte"].value_counts())
print()

print("Cobertura real de clasificaciones: solo", clasif["id_evento"].nunique(),
      "de", len(eventos), "eventos del catálogo",
      f"({clasif['id_evento'].nunique() / len(eventos):.1%})")
print()

print("Por evento de la muestra: filas y cuántas tienen SEX informado")
print(clasif.groupby("id_evento").agg(filas=("SEX", "size"), sex_informado=("SEX", "count")))

Valores nulos por columna (catálogo de eventos):
id                  0
nombre_carrera      0
fecha_inicio        0
fecha_fin           0
tipo_deporte        0
distancias        640
ubicacion         128
region            672
pais                0
codigo_pais         0
lat                 0
lon                 0
url_evento          0
municipio          60
comarca           511
provincia          33
dtype: int64

Filas completamente duplicadas (catálogo): 0
Filas duplicadas por id: 0

tipo_deporte:
tipo_deporte
Running                  203
Bicicleta de montaña     129
Carreras de montaña      107
Otros                    104
Ciclismo                  57
Carrera de Obstáculos     57
Triatlón                  30
Natación                  26
Cicloturismo              16
Carrera de fitness        11
Deportes motor             5
Ciclocross                 3
BMX                        3
Acuatlón                   3
Marcha atlética            1
Atletismo                  1
Motocross            

### Cómo sacamos `finisher_d`/`finisher_h` de las clasificaciones

RaceResult no trae un recuento de finishers por sexo ya hecho (a diferencia de ccnorte, que lo calcula el propio scraper) — aquí cada fila de `raceresult_clasificaciones.csv` es una entrada individual de la tabla de clasificación, así que hay que contar fila a fila. Usamos dos niveles, de más a menos fiable:

1. **Columna `SEX`** (`m`/`f`), cuando está informada.
2. **Marca explícita en `grupo`** (la categoría/fila de resultado, p. ej. `"#1_Hombres"`, `"#7_FEM CADETE"`, `"Cadete M"`, `"IND MAS"`, o incluso una fila de resultado completa que termina en el sexo: `"... GARCÍA TEIRA, RAMÓN M"`). Se busca tanto la palabra completa (español/inglés) como un código de una letra/abreviatura al final de la cadena.

**FIX — dos bugs que hacían el recuento poco fiable:**

- *Bug de los guiones bajos*: en regex, `_` cuenta como carácter de palabra, así que `\bhombres\b` nunca encontraba "Hombres" en `"#1_Hombres"` (no hay límite de palabra entre `_` y `H`). Se normalizan `_`/`-` a espacio antes de buscar.
- *Vocabulario incompleto*: solo se buscaban las palabras completas "femenino"/"masculino"/"mujer"/"hombre"/"varón", así que categorías en inglés (`Men`, `Women`, `Male`, `Female`) o abreviadas (`Cadete M`, `Máster 30 F`, `IND MAS`, `EQ2 MAS`) se quedaban sin detectar.

**Se elimina el antiguo "nivel 3" (convención "sin marca de sexo → masculino")**, que asumía que, en un evento donde ALGUNA categoría llevaba marca "FEM" explícita, el resto de categorías sin marca eran masculinas. Esa convención es real en algunos formatos de carrera popular española (categorías por edad sin marca = masculino, con "FEM." = femenino — es el caso del evento 419334 que motivó la regla), pero aplicada a TODOS los eventos generaba asignaciones claramente erróneas: categorías `"Team Co-Ed"`/`"Co-Ed"` (equipos mixtos por definición) o categorías de clasificación general sin split por sexo (`"4 PICOS"`, `"2 PICOS"`, `"Dist. 1700 metros"`) se contaban como 100% masculinas solo por no llevar marca — más de 15.000 filas en la muestra actual. Con el vocabulario ampliado, la mayoría de las categorías que sí llevan marca real (`"Senior - M"`, `"Máster 30 F"`...) ya se detectan directamente en el nivel 2; las que de verdad no dan ninguna pista se cuentan en `finisher_desconocido`, en vez de asumirse a ciegas — más fiel al criterio que el propio notebook ya declaraba: "nunca repartir a ciegas entre d/h".

In [3]:
import re

# Normalizamos guiones y guiones bajos a espacio ANTES de aplicar \b: en
# regex "_" cuenta como carácter de palabra, así que expresiones como
# \bhombres\b nunca encontraban "Hombres" en textos con el prefijo típico
# de RaceResult ("#1_Hombres") -- no hay límite de palabra entre "_" y "H".
def _normaliza_grupo(texto):
    return re.sub(r"[_\-]+", " ", str(texto)).strip()


_RE_FEM = re.compile(r"\bfem(enin[oa])?\b|\bmujer(es)?\b|\bfemale\b|\bwomen\b|\bwoman\b", re.IGNORECASE)
_RE_MASC = re.compile(r"\bmasculin[oa]\b|\bhombres?\b|\bvar[oó]n(es)?\b|\bmale\b|\bmen\b", re.IGNORECASE)
# RaceResult también marca el sexo como último "token" suelto de la cadena,
# tanto en categorías cortas ("Cadete M", "Máster 30 F", "IND MAS", "EQ2
# MAS") como en filas de resultado completas que terminan en el sexo del
# corredor ("... GARCÍA TEIRA, RAMÓN M"). Se ancla al final para no
# confundirlo con una inicial en medio de un nombre.
_RE_SUFIJO_F = re.compile(r"\b(f|fem)\.?\s*$", re.IGNORECASE)
_RE_SUFIJO_M = re.compile(r"\b(m|mas|masc)\.?\s*$", re.IGNORECASE)


def _sexo_por_texto(texto):
    if pd.isna(texto):
        return None
    t = _normaliza_grupo(texto)
    if _RE_FEM.search(t):
        return "d"
    if _RE_MASC.search(t):
        return "h"
    if _RE_SUFIJO_F.search(t):
        return "d"
    if _RE_SUFIJO_M.search(t):
        return "h"
    return None


def _sexo_por_sex_o_grupo(sex, grupo):
    """Nivel 1: columna SEX. Nivel 2: marca explícita (palabra o código de
    sexo) en 'grupo'. Ya no hay nivel 3 (ver celda de arriba): las filas
    sin ninguna de las dos señales se cuentan en finisher_desconocido."""
    if pd.notna(sex):
        s = str(sex).strip().lower()
        if s == "f":
            return "d"
        if s == "m":
            return "h"
    return _sexo_por_texto(grupo)


clasif["_sexo"] = [
    _sexo_por_sex_o_grupo(sex, grupo) for sex, grupo in zip(clasif["SEX"], clasif["grupo"])
]

print("Resultado de la inferencia de sexo, fila a fila:")
print(clasif["_sexo"].value_counts(dropna=False))
print()

conteos = (
    clasif.groupby(["id_evento", "_sexo"]).size().unstack(fill_value=0)
    .reindex(columns=["d", "h"], fill_value=0)
    .rename(columns={"d": "finisher_d", "h": "finisher_h"})
)
conteos["finisher_desconocido"] = (
    clasif.groupby("id_evento").size() - conteos[["finisher_d", "finisher_h"]].sum(axis=1)
)
conteos = conteos.reset_index().rename(columns={"id_evento": "id"})
conteos

Resultado de la inferencia de sexo, fila a fila:
_sexo
NaN    61542
h      20282
d       7551
Name: count, dtype: int64



_sexo      id  finisher_d  finisher_h  finisher_desconocido
0        1418           0          82                   0.0
1       97872          84         141                 225.0
2      101182         208         444                 326.0
3      101221          52         110                  81.0
4      101238          96         346                 726.0
..        ...         ...         ...                   ...
64     415278           3          50                   0.0
65     417859          72         146                 146.0
66     418766           2           0                 202.0
67     419334          18           0                  74.0
68     420299           7          93                   0.0

[69 rows x 4 columns]

In [4]:
# Unimos el recuento de sexo (solo disponible para 6 eventos) con el
# catálogo completo de 759. merge "left" deja NaN (no 0) en finisher_d/
# finisher_h/finisher_desconocido para los eventos sin clasificación
# escrapeada todavía -- la distinción "no sabemos" vs. "cero" es importante
# para no falsear ningún análisis o gráfico en Power BI.
curses_limpio = eventos.merge(conteos, on="id", how="left")
curses_limpio["tiene_clasificacion"] = curses_limpio["id"].isin(clasif["id_evento"].unique())

print("Eventos con tiene_clasificacion=True:", curses_limpio["tiene_clasificacion"].sum())
curses_limpio[["id", "nombre_carrera", "tiene_clasificacion", "finisher_d", "finisher_h", "finisher_desconocido"]].head(10)

Eventos con tiene_clasificacion=True: 147


       id  ... finisher_desconocido
0  325768  ...                792.0
1  377397  ...                  NaN
2  403612  ...                  NaN
3  263835  ...                  NaN
4  420299  ...                  0.0
5  419334  ...                 74.0
6  418933  ...                  NaN
7  418770  ...                  NaN
8  418766  ...                202.0
9  418677  ...                  NaN

[10 rows x 6 columns]

In [5]:
# "distancias" trae varias distancias por evento separadas por comas
# ("5K,10K,21K") -- nos quedamos con la más larga como distancia
# representativa del evento, igual de criterio que usar la distancia
# "principal" en el resto de fuentes.
#
# FIX: con esto solo, 639 de 759 eventos se quedaban sin distancia (0) --
# muchos más de los que en realidad "no dicen la distancia". El motivo es
# que "distancias" es un campo aparte que RaceResult no siempre rellena,
# pero el propio nombre_carrera a menudo SÍ la deja clara en texto
# ("Milla Popular La Santa", "DEKA MILE: ...", "Maratón BTT Sierra de
# Cazorla"...), igual que "Mitja"/"Marató"/"Milla" en Limpieza_xipgroc.ipynb.
# Añadimos un segundo paso por palabra clave sobre nombre_carrera, que
# solo se aplica cuando "distancias" no ha dado ningún valor (nunca
# pisa un valor ya extraído del campo estructurado).
def _distancia_maxima_km(texto):
    if pd.isna(texto):
        return None
    valores = re.findall(r"(\d+(?:[.,]\d+)?)\s*[kK]", str(texto))
    if not valores:
        return None
    return max(float(v.replace(",", ".")) for v in valores)


_RE_MEDIA_MARATON = re.compile(r"\bmedi[oa]\s+marat[oó]n\b", re.IGNORECASE)
_RE_MARATON = re.compile(r"marat[oó]n", re.IGNORECASE)
_RE_MILLA = re.compile(r"\bmillas?\b|\bmile\b", re.IGNORECASE)


def _distancia_por_nombre_km(nombre):
    """Respaldo por palabra clave en nombre_carrera cuando 'distancias' no informa nada."""
    texto = "" if pd.isna(nombre) else str(nombre)
    if _RE_MEDIA_MARATON.search(texto):
        return 21.097
    if _RE_MARATON.search(texto):
        return 42.195
    if _RE_MILLA.search(texto):
        return 1.609
    return None


curses_limpio["distancia"] = curses_limpio["distancias"].apply(_distancia_maxima_km)
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta, "distancia"] = curses_limpio.loc[_falta, "nombre_carrera"].apply(_distancia_por_nombre_km)
curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)
curses_limpio = curses_limpio.drop(columns=["distancias"])

print("Distancia informada (> 0):", (curses_limpio["distancia"] > 0).sum(), "de", len(curses_limpio),
      f"({(curses_limpio['distancia'] > 0).mean():.1%})")
print("(antes de añadir el respaldo por palabra clave en el nombre: 116 filas, 15.3%)")
curses_limpio[["nombre_carrera", "distancia"]].sample(10, random_state=0)

Distancia informada (> 0): 186 de 759 (24.5%)
(antes de añadir el respaldo por palabra clave en el nombre: 116 filas, 15.3%)


                      nombre_carrera  distancia
747                          Attrion      0.000
583                 Haria Titan 2021     21.000
575    Carrera Navidad Villanueva 22     10.000
40   DEKA MILE: Hiberus training box      1.609
243       MONTEFARO ENDURO RACE 2025      0.000
389            Sierra Cazorla Trails      0.000
428                 Haría Titan 2023     42.000
122       DEKA MILE: Box Siete Picos      1.609
31     II Carrera Popular Valdepeñas      0.000
283                   AthlonX Bilbao      0.000

In [6]:
# "tipo_deporte" ya viene en español con 19 valores directos de la fuente
# -- igual que "esport" en mychip o "Modalitat" en championsxip --, así
# que basta con un diccionario de traducción a las categorías comunes. Los
# deportes de motor/nieve (ajenos a running-ciclismo-natación-triatlón,
# el foco del TFM) y "Otros" van a "Otros".
_DEPORTE_A_TIPO = {
    "Running": "road running",
    "Carrera de fitness": "road running",
    "Marcha atlética": "marcha",
    "Atletismo": "road running",
    "Carreras de montaña": "trail running",
    "Carrera de Obstáculos": "trail running",
    "Bicicleta de montaña": "Ciclismo y btt",
    "Ciclismo": "Ciclismo y btt",
    "Cicloturismo": "Ciclismo y btt",
    "Ciclocross": "Ciclismo y btt",
    "BMX": "Ciclismo y btt",
    "Triatlón": "Multidisciplina",
    "Duatlón": "Multidisciplina",
    "Acuatlón": "Multidisciplina",
    "Natación": "Otros",
    "Deportes motor": "Otros",
    "Motocross": "Otros",
    "Snowboard": "Otros",
    "Otros": "Otros",
}
curses_limpio["tipo_modalidad"] = curses_limpio["tipo_deporte"].map(_DEPORTE_A_TIPO).fillna("Otros")

# FIX: "tipo_deporte" es la categoría que da la propia RaceResult, y no
# siempre acierta -- varias carreras con "trail" en el nombre vienen
# etiquetadas como "Running" en vez de "Carreras de montaña", así que se
# colaban como "road running" (32 de 759 filas). Mismo tipo de fallo que
# el "btt"/"ciclismo" corregido en Limpieza_sportmaniacs/
# Limpieza_cruzandolameta y documentado en Limpieza_outliers.ipynb: se
# añade una comprobación por palabra clave en nombre_carrera que
# reclasifica a "trail running" cuando el mapeo directo había dado
# "road running".
_RE_TRAIL = re.compile(r"\btrail\b", re.IGNORECASE)
_es_trail_por_nombre = curses_limpio["nombre_carrera"].fillna("").str.contains(_RE_TRAIL)
_mal_clasificada = (curses_limpio["tipo_modalidad"] == "road running") & _es_trail_por_nombre

print("Carreras con 'trail' en el nombre reclasificadas de 'road running' a 'trail running':",
      _mal_clasificada.sum())
curses_limpio.loc[_mal_clasificada, "tipo_modalidad"] = "trail running"

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("tipo_deporte original que ha caído en 'Otros':")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "tipo_deporte"].value_counts())

Carreras con 'trail' en el nombre reclasificadas de 'road running' a 'trail running': 32
tipo_modalidad
Ciclismo y btt     208
trail running      196
road running       183
Otros              137
Multidisciplina     34
marcha               1
Name: count, dtype: int64

tipo_deporte original que ha caído en 'Otros':
tipo_deporte
Otros             104
Natación           26
Deportes motor      5
Motocross           1
Snowboard           1
Name: count, dtype: int64


In [7]:
# "publico": a diferencia de ccnorte/mychip, el catálogo de RaceResult es
# una fila por EVENTO, no por categoría -- así que, en principio, solo
# podíamos clasificar por palabras clave en el propio nombre_carrera.
#
# FIX: contrastar esto con las categorías reales (columna 'grupo' de
# raceresult_clasificaciones.csv, disponible para 147 de los 759 eventos)
# muestra que es una simplificación demasiado optimista: 44 de los 143
# eventos etiquetados "Absoluta/General" por el nombre en realidad tienen
# corredores en categorías Infantil/Veteranos/Elite a la vez -- el nombre
# del evento no dice nada sobre qué categorías por edad se disputan dentro
# (p. ej. "COPA ESPAÑA PUMPTRACK..." o "XCO AJEI 2026" juntan infantil,
# absoluta, máster y élite bajo un único nombre).
#
# Para los eventos donde SÍ tenemos las clasificaciones reales, derivamos
# "publico" a partir de las categorías (`grupo`) que de verdad corrieron:
# si aparece más de un tipo de público se marca "Mixta" (el caso más
# habitual). Para el resto (612 eventos sin clasificación todavía)
# seguimos con el respaldo por nombre, la única señal disponible hasta que
# se complete el scraping.
def _clasificar_publico_texto(texto):
    t = "" if pd.isna(texto) else str(texto).lower()
    if re.search(r"\bequipos?\b|\bequips?\b|\bteam\b|\brelevos\b", t):
        return "Equipos"
    if re.search(r"discap|invident|handbike|silla de ruedas|adaptad", t):
        return "Otros"
    if re.search(r"\belite\b|profesional", t):
        return "Elite"
    if re.search(r"veteran|master|m[aá]ster|\bsenior\b", t):
        return "Mayores/Veteranos"
    if re.search(
        r"infant|benjam|alev|prebenjam|chupet|peque|menores|escolar|"
        r"cadete|juvenil|junior|j[uú]nior|promesa|sub\s?\d\d?",
        t,
    ):
        return "Infantil"
    return None


def _publico_por_categorias_reales(grupos):
    """A partir de las categorías reales de un evento (columna 'grupo'),
    devuelve 'Mixta' si aparece más de un tipo de público, el tipo único
    si solo aparece uno, 'Absoluta/General' si hay categorías pero
    ninguna con marca, o None si el evento no tiene ninguna fila."""
    tags = set()
    hubo_categoria = False
    for g in grupos.dropna().unique():
        hubo_categoria = True
        tag = _clasificar_publico_texto(_normaliza_grupo(g))
        if tag:
            tags.add(tag)
    if not hubo_categoria:
        return None
    if len(tags) >= 2:
        return "Mixta"
    if len(tags) == 1:
        return next(iter(tags))
    return "Absoluta/General"


_publico_real = clasif.groupby("id_evento")["grupo"].apply(_publico_por_categorias_reales)
_publico_por_nombre = curses_limpio["nombre_carrera"].apply(
    lambda n: _clasificar_publico_texto(n) or "Absoluta/General"
)
curses_limpio["publico"] = curses_limpio["id"].map(_publico_real).fillna(_publico_por_nombre)

print("publico (categorías reales cuando existen; si no, por nombre del evento):")
print(curses_limpio["publico"].value_counts())
print()
_tenia_real = curses_limpio["id"].map(_publico_real).notna()
_cambio = _tenia_real & (curses_limpio["publico"] != _publico_por_nombre)
print("Eventos donde la categoría real corrige lo que sugería el nombre (p. ej. a 'Mixta'):", _cambio.sum())

publico (categorías reales cuando existen; si no, por nombre del evento):
publico
Absoluta/General     683
Mixta                 36
Infantil              24
Equipos               12
Otros                  2
Mayores/Veteranos      1
Elite                  1
Name: count, dtype: int64

Eventos donde la categoría real corrige lo que sugería el nombre (p. ej. a 'Mixta'): 54


### Esquema común entre las fuentes

Para poder comparar o concatenar directamente las tablas de todas las fuentes del proyecto, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante (`"RaceResult"`). `municipio`/`comarca`/`provincia` ya venían resueltos del propio scraper (geocodificación hecha en `scrape_raceresult.ipynb`), así que no hace falta repetirla aquí. Lo propio de RaceResult (`id`, `tipo_deporte`, `url_evento`, `tiene_clasificacion`, `finisher_desconocido` y el resto de columnas de ubicación/región que no usan las demás fuentes) va al final.

In [8]:
curses_limpio["fuente"] = "RaceResult"
curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha_inicio"], errors="coerce")

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

_COLUMNAS_COMUNES = [
    "fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
    "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
]
_COLUMNAS_PROPIAS = [c for c in curses_limpio.columns if c not in _COLUMNAS_COMUNES]
curses_limpio = curses_limpio[_COLUMNAS_COMUNES + _COLUMNAS_PROPIAS]
curses_limpio.columns.tolist()

['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'id', 'fecha_inicio', 'fecha_fin', 'tipo_deporte', 'ubicacion', 'region', 'pais', 'codigo_pais', 'lat', 'lon', 'url_evento', 'finisher_desconocido', 'tiene_clasificacion']

In [9]:
# Vista final de la tabla ya limpia y clasificada.
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
print("Recordatorio de cobertura real: finisher_d/finisher_h solo están informados en",
      curses_limpio["tiene_clasificacion"].sum(), "de", len(curses_limpio),
      "eventos (el resto espera el scraping completo de clasificaciones, a correr en local).")
curses_limpio.sample(10, random_state=0)

Filas x columnas: (759, 25)

fuente                             str
nombre_carrera                     str
fecha                   datetime64[us]
dia_semana                         str
distancia                      float64
tipo_modalidad                     str
publico                            str
finisher_d                     float64
finisher_h                     float64
municipio                          str
comarca                            str
provincia                          str
id                               int64
fecha_inicio                       str
fecha_fin                          str
tipo_deporte                       str
ubicacion                          str
region                             str
pais                               str
codigo_pais                        str
lat                            float64
lon                            float64
url_evento                         str
finisher_desconocido           float64
tiene_clasificacion               b

         fuente  ... tiene_clasificacion
747  RaceResult  ...               False
583  RaceResult  ...               False
575  RaceResult  ...               False
40   RaceResult  ...                True
243  RaceResult  ...               False
389  RaceResult  ...               False
428  RaceResult  ...               False
122  RaceResult  ...               False
31   RaceResult  ...                True
283  RaceResult  ...               False

[10 rows x 25 columns]

In [10]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/raceresult/DF_RACERESULT_LIMPIO.csv")
SALIDA.parent.mkdir(parents=True, exist_ok=True)
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en ../../data/processed/raceresult/DF_RACERESULT_LIMPIO.csv
